#### What are Recommender system?
Recommender Systems are tools that suggest items to users based on their behaviour, preferences or past interactions. They help users find relevant products, movies, songs or content without manually searching for them.
#### What are main types?
 Content-Based Filtering: Content-based filtering recommends items similar to those a user liked earlier by analyzing item features and user preference profiles.
 Collaborative Filtering: Collaborative filtering identifies similarities in user behavior and recommends items based on patterns derived from many users.
 Hybrid systems combine collaborative and content-based methods to offer more accurate and robust recommendations.
 Knowledge-based systems use explicit domain knowledge and user requirements rather than historical behavior to suggest items.
 Context-Aware Recommendation Systems: These systems use contextual information such as time, location, device or mood to personalize recommendations.
#### What's cold start problem?
when a recommendation system cannot provide accurate suggestions because it lacks sufficient historical data
user cold start, item cold start, and system cold start

#### Netflix Cometision?
The Netflix Prize was a landmark machine learning competition (2006–2009) where Netflix offered a $1 million grand prize to the first team that could improve its in-house recommendation system, Cinematch, by at least 10% in predictive accuracy.

In [6]:
import pandas as pd
# Load user data

u_cols = ['user_id', 'age', 'sex', 'occupation', 'zip_code']
users = pd.read_csv('/Users/sukitharathnayake/CodeRepo/ST4035/lecture notebooks/ml-100k/u.user',
                    sep='|',
                    names=u_cols)
# Load movie data
r_cols = ['user_id', 'movie_id','rating', 'unix_timestamp']
ratings = pd.read_csv('/Users/sukitharathnayake/CodeRepo/ST4035/lecture notebooks/ml-100k/u.data',
                      sep='\t',
                      names=r_cols)

m_cols = ['movie_id', 'title',
          'release_date']
movies = pd.read_csv('/Users/sukitharathnayake/CodeRepo/ST4035/lecture notebooks/ml-100k/u.item',
                      sep='|',
                      names=m_cols,
                      usecols=range(3),
                      encoding='latin-1')
# Create a DataFrame using only the fields required
data = pd.merge(pd.merge(ratings, users), movies)
data = data[['user_id', 'title', 'movie_id', 'rating']]
print("The BD has " + str(data.shape[0]) + " ratings")
print("The BD has ", data.user_id.nunique(), " users")
print("The BD has ", data.movie_id.nunique(), " items")
print(data.head())

The BD has 100000 ratings
The BD has  943  users
The BD has  1682  items
   user_id                       title  movie_id  rating
0      196                Kolya (1996)       242       3
1      186    L.A. Confidential (1997)       302       3
2       22         Heavyweights (1994)       377       1
3      244  Legends of the Fall (1994)        51       2
4      166         Jackie Brown (1997)       346       1


In [7]:
from sklearn.model_selection import train_test_split

data_train, data_test = train_test_split(data, test_size=0.25, random_state=100)

In [8]:
# dataframe with the data from user 1
df_usr1 = data_train[data_train.user_id == 1]
# dataframe with the data from user 2
df_usr2 = data_train[data_train.user_id == 6]
# We first compute the set of common movies
common_mov = set(df_usr1.movie_id ).intersection(
df_usr2.movie_id)
print("\nNumber of common movies:", len(common_mov))


Number of common movies: 57


In [9]:
# Sub -dataframe with only the common movies
mask = ( df_usr1.movie_id. isin( common_mov))
df_usr1 = df_usr1[ mask]
print(df_usr1[[ 'title', 'rating' ]].head())
mask = ( df_usr2.movie_id. isin( common_mov))
df_usr2 = df_usr2[ mask]
print(df_usr2[[ 'title', 'rating' ]].head())

                             title  rating
34026   Princess Bride, The (1987)       5
305        Grand Day Out, A (1992)       3
58377  Fish Called Wanda, A (1988)       3
82786    Back to the Future (1985)       5
25138       Full Monty, The (1997)       5
                                        title  rating
42728  Monty Python and the Holy Grail (1974)       4
8272             2001: A Space Odyssey (1968)       5
63885                  Cinema Paradiso (1988)       4
12242             Hudsucker Proxy, The (1994)       4
32625                        Big Night (1996)       5


In [11]:
from scipy.spatial.distance import euclidean

# Similarity based on Euclidean distance for users 1-2
def SimEuclid(df, User1, User2, min_common_items=10):
    # GET MOVIES OF USER1
    mov_u1 = df[df['user_id'] == User1]
    # GET MOVIES OF USER2
    mov_u2 = df[df['user_id'] == User2]
    # FIND SHARED FILMS
    rep = pd.merge(mov_u1, mov_u2, on='movie_id')
    if len(rep) == 0:
        return 0
    if len(rep) < min_common_items:
        return 0
    return 1.0 / (1.0 + euclidean(rep['rating_x'], rep['rating_y']))


In [12]:
from scipy.stats import pearsonr
# Similarity based on Pearson correlation for user 1-2
def SimPearson(df , User1 , User2 , min_common_items = 10):
# GET MOVIES OF USER1
    mov_u1 = df[df['user_id'] == User1 ]
    # GET MOVIES OF USER2
    mov_u2 = df[df['user_id'] == User2 ]
    # FIND SHARED FILMS
    rep = pd.merge( mov_u1 , mov_u2 , on = 'movie_id')
    if len(rep)==0:
        return 0
    if(len(rep) < min_common_items):
        return 0
    return pearsonr(rep['rating_x'], rep['rating_y']) [0]

In [13]:
print("Euclidean similarity", SimEuclid(data_train, 1, 8))
print("Pearson similarity", SimPearson(data_train, 1, 8))
print("Euclidean similarity", SimEuclid(data_train, 1, 31))
print("Pearson similarity", SimPearson(data_train, 1, 31))

Euclidean similarity 0.18660549686337075
Pearson similarity 0.6808765614234659
Euclidean similarity 0
Pearson similarity 0


In [17]:
import numpy as np

def assign_to_set(df):
    sampled_ids = np.random.choice(
        df.index,
        size=int(np.ceil(len(df) * 0.2)),
        replace=False)
    df.loc[sampled_ids, 'for_testing'] = True
    return df

# Mark a random 20% of each user's rows for testing
data['for_testing'] = False
grouped = data.groupby('user_id', group_keys=False).apply(assign_to_set)
X_train = data[grouped['for_testing'] == False]
X_test = data[grouped['for_testing'] == True]
print('Train set size:', len(X_train))
print('Test set size:', len(X_test))

Train set size: 79619
Test set size: 20381


/var/folders/3f/l68d1j3x3598k23pcnmc2c440000gn/T/ipykernel_46479/2580981776.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = data.groupby('user_id', group_keys=False).apply(assign_to_set)


In [18]:
def compute_rmse(y_pred , y_true):
    """ Compute Root Mean Squared Error. """
    return np.sqrt(np.mean (np.power(y_pred - y_true , 2)))

In [21]:
class CollaborativeFiltering:
    """CF using a custom sim(u, u)."""

    def __init__(self, df, similarity=SimPearson):
        """Constructor."""
        self.sim_method = similarity
        self.df = df
        self.sim = pd.DataFrame(
            np.zeros((len(df.user_id.unique()), len(df.user_id.unique()))),
            columns=df.user_id.unique(),
            index=df.user_id.unique())

    def fit(self):
        """Prepare data structures for estimation.
        Similarity matrix for users."""
        allUsers = set(self.df['user_id'])
        self.sim = {}
        for person1 in allUsers:
            self.sim.setdefault(person1, {})
            a = self.df[self.df['user_id'] == person1][['movie_id']]
            data_reduced = pd.merge(self.df, a, on='movie_id')
            for person2 in allUsers:
                # Avoid self-comparison
                if person1 == person2:
                    continue
                self.sim.setdefault(person2, {})
                if person2 in self.sim and person1 in self.sim[person2]:
                    continue  # since symmetric matrix
                sim = self.sim_method(data_reduced, person1, person2)
                if sim < 0:
                    self.sim[person1][person2] = 0
                    self.sim[person2][person1] = 0
                else:
                    self.sim[person1][person2] = sim
                    self.sim[person2][person1] = sim

    def predict(self, user_id, movie_id):
        rating_num = 0.0
        rating_den = 0.0
        allUsers = set(self.df['user_id'])
        for other in allUsers:
            if user_id == other:
                continue
            if other not in self.sim.get(user_id, {}):
                continue
            other_rating = self.df[
                (self.df['user_id'] == other) &
                (self.df['movie_id'] == movie_id)
            ]['rating']
            if other_rating.empty:
                continue
            rating_num += self.sim[user_id][other] * float(other_rating.iloc[0])
            rating_den += self.sim[user_id][other]
        if rating_den == 0:
            movie_mean = self.df[self.df['movie_id'] == movie_id]['rating'].mean()
            if not np.isnan(movie_mean):
                return movie_mean
            return self.df[self.df['user_id'] == user_id]['rating'].mean()
        return rating_num / rating_den


In [23]:
def evaluate(fit_f, train, test):
    """RMSE-based predictive performance evaluation with pandas."""
    ids_to_estimate = zip(test.user_id, test.movie_id)
    estimated = np.array([
        fit_f(u, i) if u in train.user_id.values else 3
        for (u, i) in ids_to_estimate
    ])
    real = test.rating.values
    return compute_rmse(estimated, real)


In [29]:
reco = CollaborativeFiltering(data_train, similarity=SimPearson)
reco.fit()
print("RMSE for Collaborative Recommender: %s" % evaluate(reco.predict, data_train, data_test))

/var/folders/3f/l68d1j3x3598k23pcnmc2c440000gn/T/ipykernel_46479/3274001981.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return pearsonr(rep['rating_x'], rep['rating_y']) [0]
/var/folders/3f/l68d1j3x3598k23pcnmc2c440000gn/T/ipykernel_46479/3274001981.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return pearsonr(rep['rating_x'], rep['rating_y']) [0]
/var/folders/3f/l68d1j3x3598k23pcnmc2c440000gn/T/ipykernel_46479/3274001981.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return pearsonr(rep['rating_x'], rep['rating_y']) [0]
/var/folders/3f/l68d1j3x3598k23pcnmc2c440000gn/T/ipykernel_46479/3274001981.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return pearsonr(rep['rating_x'], rep['rating_y']) [0]
/var/folders/3f/l68d1j3x3598k23pcnmc2c440000gn/T/ipykernel_46479/327

KeyboardInterrupt: 